# NB2 — Train/Test Split (revised v2: household-grouped)

**What this notebook does:** creates the held-out test set once. After this notebook, `lasi_test.csv` is not used for any
modelling decision (feature choice, hyper-parameters, calibration) — only for the final evaluation in NB4/NB5.

**Input:** `lasi_model_ready.csv` (NB1). **Outputs:** `lasi_train.csv` (~70%), `lasi_test.csv` (~30%).

## What changed versus the previous version, and why
- **Split by household, stratified by outcome** (`StratifiedGroupKFold`), instead of a plain stratified person-level split.
  Members of one household share diet, environment, and often their reported family history. In the old split ~54% of test people had a
  household member in the training set. In our own check this did **not** change the AUC materially (≈0.73 either way), so this is a
  correctness/defensibility fix rather than a performance fix — but a reviewer can no longer raise it.
- Only the primary outcome (`chronic_heart_disease`) is used to stratify. Rows with an unknown primary outcome are dropped.
- Because the grouping changes which people are in the test set, **the new test set differs from the one used in earlier runs**.

## Assumptions (stated explicitly)
- `household_id` (from NB1) correctly identifies households (inferred from the ID structure; see NB1).
- A 70/30 split is obtained by taking 3 of the 10 folds of `StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)` as the test set. Fold sizes are therefore only *approximately* 30% because whole households are kept together.
- `random_state=42` is fixed so the split is reproducible. **Do not change it** after modelling has started.
- The same split is used for the secondary outcome (`ihd_composite`); rows with an unknown secondary outcome are dropped at analysis time, not here.

## Limitations
- This is **internal** validation: train and test come from the same survey, the same period, the same sampling design. It says nothing about other populations or time periods. NB4 adds a leave-states-out analysis as a stronger (but still internal) check.
- No survey weights are available in the merged file, so the split and every later estimate are unweighted.
- **Disclosure for the paper:** an earlier version of this pipeline was already run on a (different) split and its test-set outputs were inspected; some of the bugs fixed in v2 were found that way. The v2 test set is a new split, but no claim of "a single, never-seen test set across the whole project history" should be made.

## 0. Environment setup

In [1]:
# ============================================================
# ENVIRONMENT SETUP (same logic as before + one optional override)
# ============================================================
import os

WINDOWS_PATH = r"C:\capstone\capstone_exec\data"
COLAB_DRIVE_PATH = "/content/drive/MyDrive/capstone_exec/data"

# Optional: set the environment variable CAPSTONE_DATA to force a data folder
# (used only so the notebooks can be executed headlessly; ignore it on your own machine).
if os.environ.get("CAPSTONE_DATA"):
    BASE_PATH = os.environ["CAPSTONE_DATA"]
    print("Using CAPSTONE_DATA override.")
elif os.path.exists("/content"):
    print("Detected a genuine hosted Colab runtime (found /content).")
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_PATH = COLAB_DRIVE_PATH
else:
    print("No /content found - treating this as a local machine (VS Code, Jupyter, etc).")
    BASE_PATH = WINDOWS_PATH

if not os.path.exists(BASE_PATH):
    fallback_candidates = [
        r"C:\capstone\capstone_exec\data",
        r"C:\capstone\possible datasets\lasi-1",
        "/content/drive/MyDrive/capstone_exec/data",
        os.path.join(os.getcwd(), "data"),
    ]
    found = next((c for c in fallback_candidates if os.path.exists(c)), None)
    if found:
        print(f"BASE_PATH did not exist, using fallback: {found}")
        BASE_PATH = found
    else:
        raise FileNotFoundError("Could not find the data folder anywhere. Checked:\n  "
                                + "\n  ".join([BASE_PATH] + fallback_candidates))

LASI_RAW = os.path.join(BASE_PATH, "lasi", "raw")
LASI_PROCESSED = os.path.join(BASE_PATH, "lasi", "processed")
MODELS_DIR = os.path.join(BASE_PATH, "..", "outputs", "models")
FIGURES_DIR = os.path.join(BASE_PATH, "..", "outputs", "figures")
TABLES_DIR = os.path.join(BASE_PATH, "..", "outputs", "tables")
for _d in (LASI_PROCESSED, MODELS_DIR, FIGURES_DIR, TABLES_DIR):
    os.makedirs(_d, exist_ok=True)

print("\nBASE_PATH:", BASE_PATH)
print("LASI_PROCESSED:", LASI_PROCESSED)

No /content found - treating this as a local machine (VS Code, Jupyter, etc).

BASE_PATH: C:\capstone\capstone_exec\data
LASI_PROCESSED: C:\capstone\capstone_exec\data\lasi\processed


## 1. Load the model-ready data (from NB1)

In [2]:
import pandas as pd, numpy as np
df = pd.read_csv(os.path.join(LASI_PROCESSED, "lasi_model_ready.csv"), dtype={"state_code": str})
print("Loaded shape:", df.shape)
print("Primary-outcome prevalence (full data):")
print(df["chronic_heart_disease"].value_counts(normalize=True, dropna=False).round(4).to_string())

Loaded shape: (66859, 22)
Primary-outcome prevalence (full data):
chronic_heart_disease
No     0.9660
Yes    0.0339
NaN    0.0002


## 2. Household-grouped, outcome-stratified split

In [3]:
from sklearn.model_selection import StratifiedGroupKFold

df_labeled = df.dropna(subset=["chronic_heart_disease"]).reset_index(drop=True)
print("Rows with a known primary outcome:", len(df_labeled), "of", len(df))

y = (df_labeled["chronic_heart_disease"] == "Yes").astype(int)
groups = df_labeled["household_id"]

sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
test_positions = []
for fold, (_, te) in enumerate(sgkf.split(df_labeled, y, groups)):
    if fold < 3:
        test_positions.extend(te)

is_test = np.zeros(len(df_labeled), dtype=bool)
is_test[test_positions] = True
train_df = df_labeled[~is_test].copy()
test_df = df_labeled[is_test].copy()
print("Train:", train_df.shape, " Test:", test_df.shape, f"(test share = {len(test_df)/len(df_labeled):.1%})")

Rows with a known primary outcome: 66848 of 66859
Train: (46748, 22)  Test: (20100, 22) (test share = 30.1%)


## 3. Validation of the split
Three checks that should all pass: (1) prevalence is almost identical in both parts, (2) no person appears in both, (3) **no household appears in both**.

In [4]:
tr_rate = (train_df["chronic_heart_disease"] == "Yes").mean()
te_rate = (test_df["chronic_heart_disease"] == "Yes").mean()
print(f"Prevalence  train: {tr_rate:.4%}   test: {te_rate:.4%}   |difference| = {abs(tr_rate-te_rate):.5f}")

print("prim_keys in BOTH sets (must be 0):  ", len(set(train_df["prim_key"]) & set(test_df["prim_key"])))
print("household_ids in BOTH sets (must be 0):", len(set(train_df["household_id"]) & set(test_df["household_id"])))

print("\nSecondary outcome (ihd_composite) prevalence among known rows:")
for name, d in [("train", train_df), ("test", test_df)]:
    k = d["ihd_composite"].dropna()
    print(f"  {name}: n={len(k)}, positives={(k=='Yes').sum()}, prevalence={(k=='Yes').mean():.4%}")

print("\nPositive cases (primary outcome): train =", int((train_df['chronic_heart_disease']=='Yes').sum()),
      " test =", int((test_df['chronic_heart_disease']=='Yes').sum()))
print("States represented: train =", train_df["state_code"].nunique(), " test =", test_df["state_code"].nunique())

Prevalence  train: 3.3927%   test: 3.3781%   |difference| = 0.00015
prim_keys in BOTH sets (must be 0):   0
household_ids in BOTH sets (must be 0): 0

Secondary outcome (ihd_composite) prevalence among known rows:
  train: n=46680, positives=1047, prevalence=2.2429%
  test: n=20073, positives=461, prevalence=2.2966%

Positive cases (primary outcome): train = 1586  test = 679
States represented: train = 36  test = 36


### How to read this output
Prevalence in the two sets should differ by well under 0.01 (they will be close but, unlike the old split, not identical to 5 decimals, because households are indivisible). Both overlap counts must be **0**. The number of positive test cases drives how narrow later confidence intervals can be — with a few hundred positives, AUC intervals of roughly ±0.02 are normal.

## 4. Save

In [5]:
train_df.to_csv(os.path.join(LASI_PROCESSED, "lasi_train.csv"), index=False)
test_df.to_csv(os.path.join(LASI_PROCESSED, "lasi_test.csv"), index=False)
print("Saved lasi_train.csv", train_df.shape, "and lasi_test.csv", test_df.shape, "to", LASI_PROCESSED)

Saved lasi_train.csv (46748, 22) and lasi_test.csv (20100, 22) to C:\capstone\capstone_exec\data\lasi\processed


## Summary and rules from here on
1. Do not inspect `lasi_test.csv` outcomes, tune anything on it, or change `random_state`.
2. All preprocessing (imputation, scaling, encoding) is learned from training data only — enforced by the pipelines in NB3.
3. Report the limitations above (internal validation only, unweighted, earlier-version disclosure).

**Next:** `nb3_lasi_model_training.ipynb`.